[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Auditoría de calidad de `titanic` y `penguins`.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se aplicará la auditoría de calidad completa a `titanic` y a `penguins`, y se generará el archivo `data/penguins_limpio.csv` que usan los *notebooks* siguientes.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Los métodos de limpieza (`isna`, `notna`, `fillna`, `interpolate`, `dropna`, `duplicated`, `drop_duplicates`, …) se explican uno por uno en [py311 · 20 Limpieza y datos faltantes](https://github.com/PythonistaMX/py311/blob/main/20_limpieza_y_datos_faltantes.ipynb), y el accesor `.str` en [py311 · 23 El accesor `str` de *pandas*](https://github.com/PythonistaMX/py311/blob/main/23_el_accesor_str_de_pandas.ipynb). Aquí se usan sin repasarlos a detalle: la sesión se dedica a **decidir** qué hacer con cada problema de calidad y a medir las consecuencias.

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3. Allá las columnas de texto aparecen como `object` y se usa `inplace=True`; aquí el texto es `str` y se **reasigna** (`df["col"] = df["col"].fillna(...)`) o se usa `df.loc[...] = ...`, como se explicó en la sección 1.1 de `03_pandas_3_esencial.ipynb`.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda leerá `penguins` y `titanic` y mostrará su forma.

In [ ]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
pinguinos.shape, titanic.shape

### Recordatorio: la auditoría de calidad.

Una auditoría de calidad es una **lista de preguntas** que se contesta siempre en el mismo orden: ¿qué tipo tiene cada columna?, ¿hay filas repetidas?, ¿las categorías están escritas de una sola forma?, ¿hay valores imposibles?, ¿cuánto falta y dónde? Cada respuesta termina en una **decisión escrita**, con el número de filas que afecta. La lista completa y los mecanismos de faltantes (MCAR, MAR y MNAR) se explicaron en `07_duplicados_e_inconsistencias` y `08_valores_faltantes`; en este *notebook* se aplican a dos *datasets* reales.

---
## 1. Ejercicio de la sesión.

### Parte A · `titanic`.

### Ejercicio 1 · Cuantifica los faltantes.

**🎯 Objetivo:** medir cuántos faltantes hay y en qué columnas. **Qué se espera aprender:** a separar las columnas con pocos faltantes de las que casi no tienen datos.

Calcula el número y el porcentaje de faltantes de `age`, `deck` y `embark_town`. ¿Qué harías con cada una a primera vista?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

> 💡 **Recuerda:** un dato puede faltar por puro azar (MCAR), por algo que **sí está anotado** en otra columna, como la clase (MAR), o por su propio valor (MNAR). «MAR» significa lo segundo, **no** «al azar». Si la proporción de faltantes cambia entre grupos, se descarta el azar puro; lo contrario (que sea azar puro) nunca se puede demostrar solo con los datos.

### Ejercicio 2 · ¿La falta de `age` depende de `class`?

**🎯 Objetivo:** perfilar la falta por grupos. **Qué se espera aprender:** a descartar MCAR cuando la proporción de faltantes cambia entre grupos observados.

Calcula la proporción de `age` faltante por `class` (y, si quieres, por `sex` y por `embark_town`). Con base en eso, ¿es plausible que sea MCAR? Haz una gráfica de barras de la proporción por clase.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda graficará el porcentaje de edades faltantes por clase (requiere `por_clase`).

In [ ]:
if "por_clase" in globals():
    ax = por_clase.mul(100).plot.bar(color="tab:orange", rot=0, figsize=(5, 3))
    ax.set(ylabel="% sin edad", xlabel="clase", title="Faltantes de age por clase")
    plt.show()

**✍️ Tu respuesta:**

_(escribe aquí)_

> 💡 **Recuerda:** rellenar todos los huecos con un solo valor hace que el grupo parezca más parejo: la dispersión se encoge y las relaciones se debilitan. Rellenar dentro de cada grupo lo reduce, pero no lo elimina. Por eso, además de rellenar, se guarda una columna que marca **qué valores fueron rellenados**.

### Ejercicio 3 · Mediana global vs. mediana por `class` y `sex`.

**🎯 Objetivo:** comparar dos imputaciones sobre la distribución. **Qué se espera aprender:** que imputar por grupo respeta mejor la forma de los datos que un valor único.

1. Crea `age_med_global` (imputando con la mediana global) y `age_med_grupo` (con la mediana por `class` y `sex`, usando `groupby(...).transform("median")`), además de un indicador `age_faltaba`.
2. Compara media, mediana y desviación estándar de `age` original y de las dos versiones imputadas.
3. Grafica las tres distribuciones. ¿Qué estrategia distorsiona menos? ¿Qué le pasa a la desviación estándar?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda graficará las tres distribuciones de edad (requiere `titanic_imp`).

In [ ]:
if "titanic_imp" in globals():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True, sharey=True)
    for ax, columna, titulo in zip(axes, ["age", "age_med_global", "age_med_grupo"],
                                   ["Original (sin faltantes)", "Mediana global", "Mediana por clase y sexo"]):
        sns.histplot(titanic_imp[columna].dropna(), bins=35, ax=ax, color="tab:blue")
        ax.set(title=titulo, xlabel="edad (años)")
    plt.tight_layout()
    plt.show()

**✍️ Tu respuesta:**

_(escribe aquí)_

### Reto 1 · `deck` y `embark_town`, y un adelanto de inferencia.

**🎯 Objetivo:** decidir qué hacer con una columna casi vacía y con una casi completa. **Qué se espera aprender:** que el porcentaje de faltantes cambia por completo la decisión.

1. Propón y aplica un tratamiento para `deck` (77% faltante) y para `embark_town` (2 filas). Justifica.
2. **Adelanto de los *notebooks* 19 y 21:** con `pd.crosstab` y `scipy.stats.chi2_contingency`, prueba si la falta de `age` es independiente de `class`. Esta prueba responde cuán raro sería el patrón observado si la falta no dependiera de la clase; aquí solo se usa como adelanto: el valor *p* se explica en `19_logica_del_contraste` y la chi cuadrada en `21_proporciones_y_mas_de_dos_grupos`. Interpreta el valor *p* con cuidado.
3. ¿La tasa de supervivencia es distinta entre quienes tienen y no tienen `age` registrada? ¿Qué implicaría para un análisis que simplemente borra esas filas?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

### Parte B · Auditoría de `penguins` y *dataset* limpio para los siguientes *notebooks*.

### Ejercicio 4 · Aplica la misma auditoría a `penguins` (el original, no `sucio`).

**🎯 Objetivo:** recorrer la lista de verificación completa en un *dataset* real. **Qué se espera aprender:** a producir una auditoría completa, como la que se hace al empezar cualquier EDA.

1. Recorre la lista de verificación: tipos, duplicados, categorías (`value_counts`), rangos y faltantes por columna y por patrón.
2. Identifica las **filas sin medidas** y las filas con **`sex` faltante**. ¿La falta de `sex` se concentra en alguna especie o isla?
3. Decide y justifica: ¿qué haces con las filas sin medidas? ¿Imputas `sex`?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### *Dataset* limpio de referencia (`data/penguins_limpio.csv`).

Para que **todos empiecen el *notebook* `10_tendencia_central_y_dispersion` con los mismos datos**, la siguiente celda aplica las decisiones acordadas y guarda el resultado. Compárala con lo que hiciste en el ejercicio 4.

Decisiones: (1) `sex` → `Male`/`Female`; (2) eliminar las filas sin ninguna medida; (3) **no** imputar `sex`; (4) tipos: `category` para las categóricas y `Float64` para aleta y masa, porque son **medidas continuas**: se anotan en milímetros y gramos enteros, pero no son conteos. Por eso en el archivo aparecen con decimal (por ejemplo, `181.0`).

* La siguiente celda construirá `penguins_limpio` y lo guardará en `data/`.

In [ ]:
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
penguins_limpio = (
    pinguinos
    .assign(sex=lambda d: d["sex"].str.title())
    .dropna(subset=medidas, how="all")
    .astype({"species": "category", "island": "category", "sex": "category",
             "flipper_length_mm": "Float64", "body_mass_g": "Float64"})
    .reset_index(drop=True)
)
penguins_limpio.to_csv(os.path.join(DATA, "penguins_limpio.csv"), index=False)
len(pinguinos), len(penguins_limpio), penguins_limpio.isna().sum().to_dict()

### Reto 2 · Impacto de las decisiones.

**🎯 Objetivo:** comparar resultados con distintas decisiones de limpieza. **Qué se espera aprender:** a reportar si una decisión cambia o no las conclusiones.

Compara la masa media por especie y sexo en `pinguinos` (original) y en `penguins_limpio`. ¿Cambió algo? ¿Por qué era de esperarse? Después calcula cuánto cambiaría la masa media por especie y sexo si se hubiera imputado `sex` con la **moda** de cada especie: ¿qué sexo "gana" artificialmente?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La **lista de verificación de calidad** que se usará en todo el curso.
* Duplicados exactos vs. por llave, y por qué un duplicado sin llave no siempre es un error (`titanic`).
* Inconsistencias: texto, categorías, códigos centinela, rangos imposibles y unidades.
* Mecanismos **MCAR / MAR / MNAR** y cómo perfilarlos.
* Tratamientos y su costo: eliminar sesga si no es MCAR; imputar con un valor único **encoge la varianza** y **atenúa las correlaciones**; imputar por grupo es mejor si la causa está observada; conviene guardar el indicador.
* Decidir antes de limpiar: cuántos faltan, de qué depende la falta, para qué se usará la variable y qué población queda después de eliminar filas.

**Práctica (en Classroom):** terminar los ejercicios y verificar que exista `data/penguins_limpio.csv`.

**Siguiente *notebook*:** `10_tendencia_central_y_dispersion.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>